<a href="https://colab.research.google.com/github/nadianaz0514-cloud/urdu-ocr-codesaviours-si26-Nadia/blob/main/SI26_Week4_Nadia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q transformers torch pillow pandas sentencepiece protobuf

from google.colab import drive
drive.mount('/content/drive')

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [ ]:
!pip install sentencepiece tiktoken

In [1]:
!pip install sentencepiece -q

In [ ]:
import os
os._exit(00)

In [2]:
!pip install -q sentencepiece protobuf
!pip install -q "transformers==4.44.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten', use_fast=False)
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')
model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!
Model parameters: 333,921,792


In [6]:
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        image = Image.open(row['image']).convert('RGB')

        encoding = self.processor(
            image,
            return_tensors='pt'
        )
        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            'pixel_values': pixel_values,
            'labels': labels
        }

In [19]:
csv_path = '/content/drive/MyDrive/UrduOCR/data2/urdu_ocr_data.csv'

dataset = UrduOCRDataset(csv_path, processor)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 81 samples
Training samples: 64
Testing samples: 17


In [20]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 16
Ready to train!


In [21]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/3
------------------------------
  Batch 0/16 | Loss: 17.8122
  Batch 10/16 | Loss: 4.8121
Epoch 1 complete | Average Loss: 6.1077

Epoch 2/3
------------------------------
  Batch 0/16 | Loss: 3.5209
  Batch 10/16 | Loss: 3.2989
Epoch 2 complete | Average Loss: 3.3061

Epoch 3/3
------------------------------
  Batch 0/16 | Loss: 2.9608
  Batch 10/16 | Loss: 3.4972
Epoch 3 complete | Average Loss: 2.9943

Training complete!


In [23]:
model.eval()
print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )
        actual_text = processor.batch_decode(
            labels, skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual: {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted: �
Actual: ج

Predicted: �
Actual: ج

Predicted: ��ہ��������������
Actual: ہوٹل کا عملہ کافی دوستانہ ہے۔ ہوٹل سے

Predicted: ��ہ��������������
Actual: خوشگوار ہے۔ ہوٹل کا عملہ دوستانہ ہے۔

Predicted: ��ہ��������������
Actual: ہوٹل میں اچھے بیڈ

Predicted: ��ہ��������������
Actual: ہوٹل کا رینٹ بہت زیادہ ہے۔

Predicted: ��ہ��������������
Actual: نے پلائی گئی۔

Predicted: �
Actual: ش

Predicted: ��ہ��������������
Actual: سٹاف نہایت دوستانہ ہے۔

Predicted: �
Actual: ش

Predicted: �
Actual: ش

Predicted: �
Actual: ر

Predicted: �
Actual: ش

Predicted: ��ہ��������������
Actual: میری بہت مدد کی۔ میں یہاں ضرور واپس آؤں گا۔

Predicted: ��ہ��������������
Actual: چھ ارب ڈالر: کارگو کی مالیت، وصول شدہ رقم نہیں

Predicted: ��ہ��������������
Actual: کمرے کے مکمل صاف اور کھانا اور منظر

Predicted: ��ہ��������������
Actual: یہاں کمرے میں چائے

Accuracy: 0.0% (0/17 correct)


In [25]:
save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')
print('You can load this model again next week without retraining')

Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining
